# arXiv: gravity then mean shift

A controlled experiment on a reproducible, balanced subset of the prepared arXiv vectors. One IMSTE fit supplies a fixed 2D starting layout and a high-dimensional feature-space KNN graph for gravity. A single interactive cell applies gravity first, then mean shift using its original fixed 2D neighborhood behavior.


In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from IPython.display import display
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'pyproject.toml').is_file() and (path / 'imste' / '__init__.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate the IMSTE repository from the notebook working directory.')
for local_path in (PROJECT_ROOT, PROJECT_ROOT / 'notebooks' / 'src'):
    local_path = str(local_path)
    if local_path in sys.path:
        sys.path.remove(local_path)
    sys.path.insert(0, local_path)

from arxiv_dataset import load_arxiv_dataset
from arxiv_imste import select_arxiv_experiment_subset
from imste import IteratedMinimumSpanningTreeEmbedder
from imste._gravity import density_gravity_layout

DATA_DIR = PROJECT_ROOT / 'data' / 'arxiv'
CACHE_DIR = DATA_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
papers, X_full, data_manifest = load_arxiv_dataset(DATA_DIR)
print(f'Loaded {len(papers):,} papers × {X_full.shape[1]:,} features')
print(f'Vector matrix: {X_full.nbytes / 1024**3:.2f} GiB')


Loaded 3,233,620 papers × 384 features
Vector matrix: 4.63 GiB


## 1. Select a balanced subset

`TARGET_PER_CLASS` is the number of papers per arXiv domain (the broad class labels used in notebook 05). Sampling is deterministic, without replacement, and capped by the smallest class.


In [2]:
TARGET_PER_CLASS = 4_000
SAMPLE_RANDOM_STATE = int(data_manifest['random_state'])

primary_category = papers['categories'].fillna('unknown').str.split().str[0]
archive = primary_category.str.split('.').str[0]
domain_names = {
    'cs': 'computer science', 'math': 'mathematics', 'eess': 'engineering',
    'stat': 'statistics', 'q-bio': 'biology', 'q-fin': 'finance', 'econ': 'economics',
}
domain = archive.map(domain_names).fillna('physics')
class_counts = domain.value_counts()
PER_CLASS = min(int(TARGET_PER_CLASS), int(class_counts.min()))
if PER_CLASS < TARGET_PER_CLASS:
    print(f'Capped at {PER_CLASS:,} papers per class (smallest class).')

labels_full = pd.Categorical(domain, categories=class_counts.index.tolist())
X_experiment, labels, subset_id = select_arxiv_experiment_subset(
    X_full, labels_full,
    dataset_id=data_manifest['dataset_id'],
    mode='balanced', sample_size=PER_CLASS * len(class_counts),
    random_state=SAMPLE_RANDOM_STATE,
)
X_experiment = np.asarray(X_experiment, dtype=np.float32)
label_values = np.asarray(labels).astype(str)
class_order = list(labels.categories)
print(f'Balanced subset: {len(X_experiment):,} papers ({PER_CLASS:,} per class), ID {subset_id}')
display(pd.Series(label_values).value_counts().sort_index().rename('papers').to_frame())


Balanced subset: 32,000 papers (4,000 per class), ID af7faf6879f3feb640fa


,papers
biology,4000
computer science,4000
economics,4000
engineering,4000
finance,4000
mathematics,4000
physics,4000
statistics,4000


## 2. Fit one shared IMSTE starting layout

The combined interactive cell starts from this fixed layout. Gravity and mean shift are applied sequentially after the fit, so changing either control does not rerun IMSTE.


In [ ]:
RANDOM_STATE = int(data_manifest['random_state'])
IMSTE_SETTINGS = {
    'n_components': 2,
    'n_msts': 3,
    'n_epochs': 200,
    'batch_size': 32_768,
    'learning_rate': 0.05,
    'negative_ratio': 1,
    'lambda_rep': 0.2,
    'random_state': RANDOM_STATE,
    'device': 'mps',  # Change to 'auto' or 'cpu' if needed.
    'mst_method': 'famst',
    'mst_neighbor_backend': 'rinnd',
    'mst_neighbors': 15,
    'mst_inter_component_edges': 5,
    'pca_init': True,
    'pca_init_scaling': 'global',
    'multilevel_init': True,
    'multilevel_reduction_ratio': 8,
    'multilevel_base_size': 512,
    'multilevel_epochs': 10,
    'metric': 'euclidean',
    'gravity_mode': None,
    'gravity_neighbors': 200,  # Cache width for the interactive gravity slider.
    # Cache the feature-space KNN graph wide enough for the interactive cells.
    'mean_shift_neighbors': 200,
    'mean_shift_iterations': 0,
    'verbose': True,
}
fit_started = time.perf_counter()
estimator = IteratedMinimumSpanningTreeEmbedder(**IMSTE_SETTINGS)
Z_base = estimator.fit_transform(X_experiment)
fit_seconds = time.perf_counter() - fit_started
Z_base = np.asarray(Z_base[:, :2], dtype=np.float32)
print(f'IMSTE starting layout: {fit_seconds:.1f} s')


## 3. Reuse the base feature-space graph

The high-dimensional KNN graph is captured during the base IMSTE fit. The combined controls reuse it for gravity and mean shift; changing mean-shift settings does not launch another neighbor search.


In [ ]:
import ipywidgets as widgets
from io import BytesIO
# Reuse the high-dimensional graph captured by the base FAMST fit. If this
# already-fitted estimator did not retain enough neighbors, build one wide graph
# here and reuse it for every gravity update. Mean shift uses its original 2D graph.
GRAVITY_GRAPH_WIDTH = min(200, len(X_experiment) - 1)
_cached_feature_graph = getattr(estimator, 'feature_neighbor_graph_', None)
if _cached_feature_graph is not None and _cached_feature_graph[0].shape[1] >= GRAVITY_GRAPH_WIDTH:
    FEATURE_NEIGHBOR_INDICES, FEATURE_NEIGHBOR_DISTANCES = _cached_feature_graph
else:
    from rinnd import RINND

    _feature_search = RINND(
        np.ascontiguousarray(X_experiment, dtype=np.float32),
        metric='euclidean',
        n_neighbors=GRAVITY_GRAPH_WIDTH,
        random_state=RANDOM_STATE,
        n_trees=8,
        n_iters=max(5, int(round(np.log2(len(X_experiment))))),
        max_candidates=max(20, min(60, int(round(1.5 * GRAVITY_GRAPH_WIDTH)))),
        n_jobs=-1,
        verbose=False,
    )
    FEATURE_NEIGHBOR_INDICES, FEATURE_NEIGHBOR_DISTANCES = _feature_search.neighbor_graph
    del _feature_search
FEATURE_NEIGHBOR_INDICES = np.asarray(FEATURE_NEIGHBOR_INDICES, dtype=np.int64)
FEATURE_NEIGHBOR_DISTANCES = np.asarray(FEATURE_NEIGHBOR_DISTANCES, dtype=np.float64)
del _cached_feature_graph

palette = plt.get_cmap('tab20')
color_key = {name: to_hex(palette(i % palette.N)) for i, name in enumerate(class_order)}
PLOT_COLORS = [color_key[label] for label in label_values]

print(
    f'Reusing {FEATURE_NEIGHBOR_INDICES.shape[1]} high-dimensional feature neighbors '
    f'for gravity on {len(Z_base):,} points.'
)


## 4. Interactive gravity → mean shift

Each update starts from the fixed IMSTE base, applies gravity, then applies mean shift to the gravity result. Both refinements use the feature-space KNN graph captured during the base fit. The plot compares the base and final layouts once. Set gravity pull or mean-shift iterations to zero to disable that effect.


In [ ]:
from imste._mean_shift import embedding_neighbor_indices, gaussian_mean_shift_layout

# Reuse the one high-dimensional feature graph prepared from the base fit.
_COMBINED_FEATURE_INDICES = FEATURE_NEIGHBOR_INDICES
_COMBINED_FEATURE_DISTANCES = FEATURE_NEIGHBOR_DISTANCES
# Preserve the original mean-shift behavior: fixed neighbor identities from the
# base 2D layout, with distances and kernel weights refreshed at each iteration.
_COMBINED_MEAN_SHIFT_INDICES = embedding_neighbor_indices(
    Z_base, min(200, len(Z_base) - 1)
)


def _mean_shift_from_cached_graph(*, neighbors, bandwidth_factor, step_size, iterations, embedding):
    shifted, _ = gaussian_mean_shift_layout(
        embedding,
        neighbor_indices=_COMBINED_MEAN_SHIFT_INDICES,
        neighbors=int(neighbors),
        bandwidth_factor=float(bandwidth_factor),
        step_size=float(step_size),
        iterations=int(iterations),
    )
    return shifted


def _combined_layout_image(base, refined, gravity_seconds, mean_shift_seconds):
    layouts = [
        (base, 'Fixed IMSTE base'),
        (refined, f'Gravity → mean shift · {gravity_seconds:.2f} s + {mean_shift_seconds:.2f} s'),
    ]
    all_points = np.vstack([coordinates for coordinates, _ in layouts])
    low = all_points.min(axis=0)
    high = all_points.max(axis=0)
    center = (low + high) / 2
    radius = max(float(np.max(high - low)) * 0.52, 1e-6)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    for ax, (coordinates, title) in zip(axes, layouts):
        ax.scatter(coordinates[:, 0], coordinates[:, 1], c=PLOT_COLORS, s=4,
                   alpha=0.65, linewidths=0, rasterized=True)
        ax.set_title(title)
        ax.set_xlim(center[0] - radius, center[0] + radius)
        ax.set_ylim(center[1] - radius, center[1] + radius)
        ax.set_aspect('equal', adjustable='box')
        ax.set_axis_off()
    image_buffer = BytesIO()
    fig.savefig(image_buffer, format='png', dpi=120)
    plt.close(fig)
    return image_buffer.getvalue()

_combined_controls = {
    'gravity_mode': widgets.Dropdown(options=['density_core', 'mexican_hat'], value='density_core', description='Gravity mode'),
    'gravity_neighbors': widgets.IntSlider(value=15, min=5, max=min(200, len(Z_base) - 1), step=5, description='Graph k', continuous_update=False),
    'gravity_resolution': widgets.FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, readout_format='.1f', description='Resolution', continuous_update=False),
    'gravity_strength': widgets.FloatSlider(value=0.35, min=0.0, max=0.9, step=0.05, readout_format='.2f', description='Gravity pull', continuous_update=False),
    'mean_neighbors': widgets.IntSlider(value=50, min=5, max=min(200, len(Z_base) - 1), step=5, description='Neighbors', continuous_update=False),
    'mean_bandwidth': widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1, readout_format='.1f', description='Bandwidth', continuous_update=False),
    'mean_step': widgets.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05, readout_format='.2f', description='Mean step', continuous_update=False),
    'mean_iterations': widgets.IntSlider(value=10, min=0, max=30, step=1, description='Mean iterations', continuous_update=False),
}
_combined_plot = widgets.Image(format='png', layout=widgets.Layout(width='100%'))
_combined_cache = {
    'gravity_key': None, 'gravity': None, 'gravity_seconds': 0.0,
    'mean_key': None, 'mean_shift': None, 'mean_seconds': 0.0,
}


def _render_combined(change=None):
    gravity_key = tuple(_combined_controls[name].value for name in (
        'gravity_mode', 'gravity_neighbors', 'gravity_resolution', 'gravity_strength',
    ))
    if gravity_key != _combined_cache['gravity_key']:
        mode, neighbors, resolution, strength = gravity_key
        started = time.perf_counter()
        if float(strength) == 0.0:
            _combined_cache['gravity'] = Z_base.copy()
        else:
            _combined_cache['gravity'], diagnostics = density_gravity_layout(
                X_experiment, Z_base,
                mode=mode,
                neighbors=int(neighbors),
                resolution=float(resolution),
                random_state=RANDOM_STATE,
                strength=float(strength),
                feature_neighbor_graph=(FEATURE_NEIGHBOR_INDICES, FEATURE_NEIGHBOR_DISTANCES),
                feature_neighbor_metric='euclidean',
            )
        _combined_cache['gravity_seconds'] = time.perf_counter() - started
        _combined_cache['gravity_key'] = gravity_key
        _combined_cache['mean_key'] = None

    mean_key = tuple(_combined_controls[name].value for name in (
        'mean_neighbors', 'mean_bandwidth', 'mean_step', 'mean_iterations',
    ))
    if mean_key != _combined_cache['mean_key']:
        neighbors, bandwidth, step, iterations = mean_key
        started = time.perf_counter()
        _combined_cache['mean_shift'] = _mean_shift_from_cached_graph(
            neighbors=neighbors,
            bandwidth_factor=bandwidth,
            step_size=step,
            iterations=iterations,
            embedding=_combined_cache['gravity'],
        )
        _combined_cache['mean_seconds'] = time.perf_counter() - started
        _combined_cache['mean_key'] = mean_key

    _combined_plot.value = _combined_layout_image(
        Z_base, _combined_cache['mean_shift'],
        _combined_cache['gravity_seconds'], _combined_cache['mean_seconds'],
    )

for _control in _combined_controls.values():
    _control.observe(_render_combined, names='value')
_render_combined()
_gravity_row = widgets.HBox([_combined_controls[name] for name in (
    'gravity_mode', 'gravity_neighbors', 'gravity_resolution', 'gravity_strength',
)])
_mean_shift_row = widgets.HBox([_combined_controls[name] for name in (
    'mean_neighbors', 'mean_bandwidth', 'mean_step', 'mean_iterations',
)])
display(widgets.VBox([
    widgets.HTML('<b>Gravity, then mean shift</b>'), _gravity_row,
    widgets.HTML('<b>Mean shift controls</b>'), _mean_shift_row,
    _combined_plot,
]))
